In [ ]:
import sys
import os
import numpy as np
import requests
import json

sys.path.insert(0, "../src/")
from schema import Chunk, Document, ChunkStore
from cadec_loader import load_cadec
from chunker import chunk_document
from embedder import load_model, encode
from retriever import retrieve, detect_drug, show_results

# Environment variables — must be first
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import faiss

In [ ]:
#load model
load_model()

# load cadec doc
all_chunks=[]
docs = load_cadec("../data/cadec")
for doc in docs:
    all_chunks.extend(chunk_document(doc, overlap=1, chunk_size=400))

# load embeddings and index
embeddings = np.load("../index/embeddings.npy")
index = faiss.read_index("../index/cadec.faiss")

# rebuild chunk store
store = ChunkStore()
for chunk in all_chunks:
    store.add(chunk)

print(f"Ready: {len(all_chunks)} chunks, {index.ntotal} vectors")

In [ ]:
def build_prompt(query: str, chunks: list)->str:
    """
    Assemble retrieved chunks + query into a structured prompt.
    
    Design decisions:
    - Role definition first: tells the LLM what it is
    - Hard constraint: ONLY use provided context
    - Explicit fallback: what to say if context is insufficient
    - Citation instruction: forces the LLM to ground its answer
    - Context before question: LLM reads evidence before seeing the question
    """

    # build context block
    context_lines= []
    for i, result in enumerate(chunks):
        chunk= result["chunk"]
        context_lines.append(
            f"[Report: {i+1} | Drug: {chunk.drug_name} | ID: {chunk.chunk_id}]\n"
            f"{chunk.text}"
        )

    context_block= "\n\n".join(context_lines)
    prompt = f"""You are a drug safety assistant helping analyse patient-reported adverse drug reactions.

    Your task is to answer the question below using ONLY the patient reports provided.
    Do not use any knowledge from your training data.
    If the provided reports do not contain enough information to answer, respond with:
    "The available patient reports do not contain sufficient information to answer this question."

    For each claim you make, cite the report number it comes from, like: (Report 1), (Report 3).

    PATIENT REPORTS:
    {context_block}

    QUESTION:
    {query}

    ANSWER:"""

    return prompt
        

In [ ]:
# test prompt builder
query = "What are the muscle side effects of atorvastatin?"
query_vector = encode([query]).astype(np.float32)

results = retrieve(query_vector, query, store, embeddings, index, k=3)

prompt = build_prompt(query, results)
print(prompt)
print(f"\n--- Prompt length: {len(prompt)} chars ---")

In [ ]:
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

In [ ]:
# ANTHROPIC_API_KEY = "your-api-key-here"  
# # Better: os.environ.get("ANTHROPIC_API_KEY")

def call_llm(prompt: str, max_tokens: int = 500) -> str:
    """
    Direct HTTP call to GROQ API.
    No SDK. No wrapper. Just requests.
    """
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {GROQ_API_KEY}"
    }
    
    body = {
        "model": "llama-3.1-8b-instant",  # free, fast, good quality
        "max_tokens": max_tokens,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }
    
    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers=headers,
        json=body
    )
    
    if response.status_code != 200:
        raise RuntimeError(f"API error {response.status_code}: {response.text}")
    
    data = response.json()
    return data["choices"][0]["message"]["content"]

In [ ]:
def answer(query: str, k: int =5)->str:
    """
    Full RAG pipeline:
    query → encode → retrieve → build prompt → call LLM → return answer
    """
    print(f"Query: {query}")
    print("="*60)

    # step 1 - encode query
    query_vector= encode([query]).astype(np.float32)

    # step 2 - retrive relevant chunks
    results = retrieve(query_vector, query, store, embeddings, index, k=k)
    print(f"Retrieved {len(results)} chunks:")
    for i, r in enumerate(results):
        print(f"  [{i+1}] {r['chunk'].text[:80]}...")
    print()

    # step 3 - build prompt
    prompt= build_prompt(query, results)

    # step 4 - call llm
    print("Calling llm...")
    answer_text = call_llm(prompt)

    print("\nAnswer:")
    print(answer_text)
    return answer_text

    
    

In [ ]:
answer("What are the muscle side effects of atorvastatin?")